# 02 · Modelling & tournament simulation

Builds match-level features from four data sources, trains XGBoost models for every quantity in the
DataCamp *Predict the FIFA World Cup 2026* competition (result, home/away goals, corners, yellow cards, red cards),
then simulates the 72 group matches and the 32-match knockout bracket.

**Inputs** (`data/`, see the README for how to obtain them): `matches_stats.csv`, `elo_ratings.csv`,
`fifa_rankings.csv`, `team_records.csv`, `group_fixtures.csv`, `knockout_slots.csv`

**Outputs** (`outputs/`): `group_predictions.csv`, `knockout_predictions.csv`

Sections: 1 Load → 2 Clean → 3 Impute → 4 Merge → 5 Features → 6 Feature selection → 7 Split →
8 Tuning → 9 Validation → 10 Final models → 11 Prediction function → 12 Group stage → 13 Knockout stage → 14 Export

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
DATA = ROOT / "data"
OUTPUTS = ROOT / "outputs"
SEED = 42

## 1. Load data and standardise team names
Each source spells some countries differently (e.g. *Korea Republic* vs *South Korea*), so names are mapped to one convention before any joins.

In [2]:
COUNTRY_NAMES = {
    "US": "USA",
    "United States": "USA",
    "United States of America": "USA",
    "United States Virgin Islands": "US Virgin Islands",

    "Turkey": "Türkiye",
    "Turkiye": "Türkiye",

    "IR Iran": "Iran",

    "Korea Republic": "South Korea",
    "Korea, South": "South Korea",
    "Republic of Korea": "South Korea",
    "Korea DPR": "North Korea",
    "Korea, North": "North Korea",

    "Ivory Coast": "Côte d'Ivoire",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Cote dIvoire": "Côte d'Ivoire",
    "Cote d Ivoire": "Côte d'Ivoire",

    "Congo DR": "DR Congo",
    "Congo, DR": "DR Congo",
    "Republic of Congo": "DR Congo",
    "Democratic Republic of the Congo": "DR Congo",

    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Bosnia-Herzegovina": "Bosnia and Herzegovina",
    "Bosnia/Herzegovina": "Bosnia and Herzegovina",
    "Bosnia": "Bosnia and Herzegovina",

    "Sao Tome & Principe": "Sao Tome and Principe",
    "São Tomé and Príncipe": "Sao Tome and Principe",

    "UAE": "United Arab Emirates",
    "China PR": "China",
    "Hong Kong, China": "Hong Kong",
    "Hong Kong China": "Hong Kong",
    "Macao": "Macau",
    "Brunei Darussalam": "Brunei",
    "Czech Republic": "Czechia",
    "Curacao": "Curaçao",
    "Cape Verde": "Cabo Verde",
    "Republic of Ireland": "Ireland",
    "Republic of North Macedonia": "North Macedonia",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Eastern Samoa": "American Samoa",
    "Central African Rep": "Central African Republic",
    "Turks and Caicos Islands": "Turks and Caicos",
    "East Timor": "Timor-Leste",
    "The Gambia": "Gambia",
    "Taiwan": "Chinese Taipei",

    "St. Kitts & Nevis": "Saint Kitts and Nevis",
    "St Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "St Lucia": "Saint Lucia",
    "St. Vincent & Grenadinens": "Saint Vincent and the Grenadines",
    "St Vincent/Gren": "Saint Vincent and the Grenadines",
    "St Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "St Vincent and Grenadines": "Saint Vincent and the Grenadines",
}

def normalize_country_name(name):
    return COUNTRY_NAMES.get(name, name)

In [3]:
elo     = pd.read_csv(DATA / "elo_ratings.csv")
fifa    = pd.read_csv(DATA / "fifa_rankings.csv")
matches = pd.read_csv(DATA / "matches_stats.csv")
h2h     = pd.read_csv(DATA / "team_records.csv")

elo["Team"] = elo["Team"].apply(normalize_country_name)
fifa["team"] = fifa["team"].apply(normalize_country_name)
matches["home_team"] = matches["home_team"].apply(normalize_country_name)
matches["away_team"] = matches["away_team"].apply(normalize_country_name)
h2h["Team"] = h2h["Team"].apply(normalize_country_name)
h2h["Country"] = h2h["Country"].apply(normalize_country_name)

print(f"Matches: {len(matches):,} rows")
print(f"Elo:     {len(elo)} rows")
print(f"FIFA:    {len(fifa)} rows")
print(f"H2H:     {len(h2h):,} rows")

Matches: 2,458 rows
Elo:     244 rows
FIFA:    211 rows
H2H:     2,359 rows


## 2. Clean and reshape each source

**Elo ratings** — keep rank and rating, and derive an all-time win rate.

In [4]:
elo = elo.rename(columns={
    "Rank": "elo_rank",
    "Team": "team",
    "Elo Rating": "elo_rating",
    "Total Matches": "elo_total_matches",
    "Wins": "elo_wins"
    })

elo["elo_win_rate"] = elo["elo_wins"] / elo["elo_total_matches"].replace({0: np.nan})

cols_to_keep = ["team", "elo_rank", "elo_rating", "elo_win_rate"]

elo = elo[[c for c in cols_to_keep if c in elo.columns]].copy()

elo.head(3)

,team,elo_rank,elo_rating,elo_win_rate
0,Spain,1,2155,0.590269
1,Argentina,2,2113,0.550045
2,France,3,2062,0.504792


**FIFA rankings**

In [5]:
fifa = fifa.rename(columns={
    "rank": "fifa_rank",
    "points": "fifa_points",
    })
fifa = fifa[["team", "fifa_rank", "fifa_points", "confederation"]].copy()
fifa.head(3)

,team,fifa_rank,fifa_points,confederation
0,France,1,1877.32,UEFA
1,Spain,2,1876.40,UEFA
2,Argentina,3,1874.81,CONMEBOL


**Matches** — standardise column names and types; drop indicator columns that duplicate the goals/result.

In [6]:
matches = matches.rename(columns={
    "home_score": "home_goals",
    "away_score": "away_goals",
    "home_team_possession": "home_poss",
    "away_team_possession": "away_poss",
    "home_team_xG": "home_xG",
    "away_team_xG": "away_xG",
    "home_team_shots": "home_shots",
    "away_team_shots": "away_shots",
    "home_YC": "home_yellows",
    "away_YC": "away_yellows",
    "home_RC": "home_reds",
    "away_RC": "away_reds",
    "home_or_away_win": "winning_team",
    "winner": "winning_team_name",
    "went_to_penalties": "penalties",
    })

matches["date"] = pd.to_datetime(matches["date"])
matches["is_competitive"] = matches["is_competitive"].astype(str).str.upper().isin(["TRUE", "1", "YES"])
matches["neutral"]        = matches["neutral"].astype(str).str.upper().isin(["TRUE", "1", "YES"])
matches = matches.sort_values("date").reset_index(drop=True)

NUMERIC_COLS = [
    "home_goals", "away_goals",
    "home_yellows", "away_yellows",
    "home_reds", "away_reds",
    "home_corners", "away_corners",
    "home_xG", "away_xG",
    "home_shots", "away_shots",
    "home_SoT", "away_SoT",
    "home_poss", "away_poss",
]
for col in NUMERIC_COLS:
    if col in matches.columns:
        matches[col] = pd.to_numeric(matches[col], errors="coerce")

# Deleting these columns as this information is redundant and derived from the goals and winning team
drop_cols = [c for c in ["is_draw", "over_2_5", "btts"] if c in matches.columns]
matches.drop(columns=drop_cols, inplace=True)
matches.head(3)

,date,tournament,home_team,away_team,is_competitive,city,country,neutral,home_goals,away_goals,...,home_shots,away_shots,home_SoT,away_SoT,home_corners,away_corners,home_yellows,away_yellows,home_reds,away_reds
0,2024-01-01,Friendly,Japan,Thailand,False,Tokyo,Japan,False,5,0,...,30.0,8.0,10.0,1.0,13.0,3.0,0.0,2.0,0.0,0.0
1,2024-01-01,Friendly,China,Hong Kong,False,Abu Dhabi,United Arab Emirates,True,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-02,Friendly,Indonesia,Libya,False,Antalya,Turkey,True,0,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Head-to-head records** — parse goals and win ratio, and derive rate features.

In [7]:
h2h = h2h.rename(columns={
    "Team": "team",
    "Country": "opponent",
    "Matches": "h2h_matches",
    "W": "h2h_wins",
    "D": "h2h_draws",
    "L": "h2h_losses",
    "Goals": "goals_str",
    "+/-": "h2h_goal_diff",
    "Points": "h2h_points",
    "PPG": "h2h_ppg",
    "Win ratio": "win_ratio_str",
    "Average attendance": "h2h_avg_att",
    })

# Convert goals string "x:y" to separate columns "for" and "against"
def parse_goals(goals_str):
    parts = str(goals_str).split(":")
    if len(parts) == 2:
        return int(parts[0]), int(parts[1])
    else:
        return np.nan, np.nan
    
h2h["h2h_goals_f"], h2h["h2h_goals_a"] = zip(*h2h["goals_str"].apply(parse_goals))
h2h["h2h_goals_f"] = pd.to_numeric(h2h["h2h_goals_f"], errors="coerce")
h2h["h2h_goals_a"] = pd.to_numeric(h2h["h2h_goals_a"], errors="coerce")
h2h["h2h_goals_f_per_match"] = h2h["h2h_goals_f"] / h2h["h2h_matches"].replace({0: np.nan})
h2h["h2h_goals_a_per_match"] = h2h["h2h_goals_a"] / h2h["h2h_matches"].replace({0: np.nan})
h2h["h2h_win_rate"] = h2h["win_ratio_str"].str.rstrip("%").astype(float) / 100.0
h2h["h2h_draw_rate"] = h2h["h2h_draws"] / h2h["h2h_matches"].replace({0: np.nan})
h2h["h2h_loss_rate"] = h2h["h2h_losses"] / h2h["h2h_matches"].replace({0: np.nan})
# Dominance: weighted mix of win rate and goal difference per match (higher = more dominant)
h2h["h2h_dominance"] = (
    h2h["h2h_win_rate"].fillna(0.5) * 0.6
    + (h2h["h2h_goal_diff"] / (h2h["h2h_matches"] * 3)).clip(-1, 1) * 0.4
)
# Taking 4 or more matches as a reliable head-to-head record (arbitrary threshold) 
h2h["h2h_reliable"] = (h2h["h2h_matches"] >= 4).astype(int)
# Dropping columns that are already captured by the new features or are redundant
h2h.drop(columns=["goals_str", "win_ratio_str"], inplace=True)
h2h.head(3)

,team,opponent,h2h_matches,h2h_wins,h2h_draws,h2h_losses,h2h_goal_diff,h2h_points,h2h_ppg,h2h_avg_att,h2h_goals_f,h2h_goals_a,h2h_goals_f_per_match,h2h_goals_a_per_match,h2h_win_rate,h2h_draw_rate,h2h_loss_rate,h2h_dominance,h2h_reliable
0,Mexico,Panama,13,11,2,0,18,35,2.69,32.511,21.0,3.0,1.615385,0.230769,0.846,0.153846,0.0,0.692215,1
1,Mexico,Honduras,15,10,2,3,19,32,2.13,38.785,28.0,9.0,1.866667,0.600000,0.667,0.133333,0.2,0.569089,1
2,Mexico,Costa Rica,10,7,3,0,9,24,2.40,33.674,21.0,12.0,2.100000,1.200000,0.700,0.300000,0.0,0.540000,1


## 3. Impute missing match statistics
The FotMob statistics (possession, xG, shots, corners, cards) are missing for most matches.
Coverage is recorded **before** imputation, together with flags for rows whose corners and cards were actually observed,
so the corners/cards models can later be evaluated only on real values.

In [8]:
STAT_COLS = ["home_poss", "home_xG", "home_shots", "home_corners", "home_yellows", "home_reds"]
raw_coverage = matches[STAT_COLS].notna().mean().mul(100).round(1)
print("Share of matches with observed values (%):")
print(raw_coverage.to_string())

matches["corners_observed"] = matches[["home_corners", "away_corners"]].notna().all(axis=1)
matches["cards_observed"]   = matches[["home_yellows", "away_yellows"]].notna().all(axis=1)

Share of matches with observed values (%):
home_poss       41.0
home_xG         16.6
home_shots      41.0
home_corners    40.9
home_yellows    40.6
home_reds       42.0


In [9]:
# Calculating team medians for nan imputation where there is some data available for a team 
def build_team_medians(df, home_col, away_col, home_team="home_team", away_team="away_team"):
    home = (df[[home_team, home_col]].rename(columns={home_team: "team", home_col: "value"}))
    away = (df[[away_team, away_col]].rename(columns={away_team: "team", away_col: "value"}))
    medians = pd.concat([home, away]).groupby("team")[("value")].median()
    return medians

def impute_stats(df, home_col, away_col, team_medians, global_median=None):
    home = df[home_col].copy()
    away = df[away_col].copy()

    mask_home = home.isna()
    mask_away = away.isna()
    home[mask_home] = home[mask_home].index.map(
        lambda idx: team_medians.get(df.loc[idx, "home_team"], global_median))
    away[mask_away] = away[mask_away].index.map(
        lambda idx: team_medians.get(df.loc[idx, "away_team"], global_median))

    both_null = home.isna() & away.isna()
    home = home.fillna(global_median)
    away = away.fillna(global_median)

    return home, away, both_null

In [10]:
# Impute cards, corners, and other stats if available
for home_col, away_col in [
    ("home_yellows", "away_yellows"),
    ("home_reds",    "away_reds"),
    ("home_corners", "away_corners"),
    ("home_xG",      "away_xG"),
    ("home_shots",   "away_shots"),
    ("home_SoT",     "away_SoT"),
]:
    if home_col not in matches.columns:
        continue
    medians = build_team_medians(matches, home_col, away_col)
    global_median = matches[[home_col, away_col]].stack().median()
    matches[home_col], matches[away_col], _ = impute_stats(
        matches, home_col, away_col, medians, global_median)

medians = build_team_medians(matches, "home_poss", "away_poss")
matches["home_poss"], matches["away_poss"], _ = impute_stats(
    matches, "home_poss", "away_poss", medians, 50.0)

matches["goals"] = matches["home_goals"] + matches["away_goals"]
matches["yellow_cards"] = matches["home_yellows"] + matches["away_yellows"]
matches["red_cards"] = matches["home_reds"] + matches["away_reds"]
matches["corners"] = matches["home_corners"] + matches["away_corners"]
matches.head(3)

,date,tournament,home_team,away_team,is_competitive,city,country,neutral,home_goals,away_goals,...,home_yellows,away_yellows,home_reds,away_reds,corners_observed,cards_observed,goals,yellow_cards,red_cards,corners
0,2024-01-01,Friendly,Japan,Thailand,False,Tokyo,Japan,False,5,0,...,0.0,2.0,0.0,0.0,True,True,5,2.0,0.0,16.0
1,2024-01-01,Friendly,China,Hong Kong,False,Abu Dhabi,United Arab Emirates,True,1,2,...,2.0,1.5,0.0,0.0,False,False,3,3.5,0.0,5.0
2,2024-01-02,Friendly,Indonesia,Libya,False,Antalya,Turkey,True,0,4,...,1.5,2.0,0.0,0.0,False,False,4,3.5,0.0,8.0


## 4. Merge ratings and head-to-head records onto each match

In [11]:
# Merge Elo and FIFA rankings for both home & away teams
def merge_side(df, ratings, side):
    prefixed = ratings.copy()
    prefixed.columns = [f"{side}_{c}" if c != "team" else c for c in prefixed.columns]
    return df.merge(prefixed.rename(columns={"team": f"{side}_team"}), how="left", left_on=f"{side}_team", right_on=f"{side}_team")

matches = merge_side(matches, elo, "home")
matches = merge_side(matches, elo, "away")
matches = merge_side(matches, fifa, "home")
matches = merge_side(matches, fifa, "away")

# Merge h2h data 
H2H_KEEP = ["team", "opponent", "h2h_matches",
            "h2h_goals_f_per_match", "h2h_goals_a_per_match",
            "h2h_goal_diff", "h2h_win_rate", "h2h_draw_rate", "h2h_loss_rate",
            "h2h_dominance", "h2h_reliable"]
h2h_trim = h2h[[c for c in H2H_KEEP if c in h2h.columns]].copy()

home_h2h = (h2h_trim
    .rename(columns={"team": "home_team", "opponent": "away_team"}))

matches = matches.merge(home_h2h, on=["home_team", "away_team"], how="left")

## 5. Feature engineering

**Rating gaps and confederation strength** (confederation strengths are hand-assigned).

In [12]:
# Assigning a confederation strength
CONF_STRENGTH = {
    "UEFA":     5,
    "CONMEBOL": 4,
    "CAF":      2.5,
    "AFC":      2.5,
    "CONCACAF": 2,
    "OFC":      1,
}
matches["home_conf_strength"] = matches["home_confederation"].map(CONF_STRENGTH).fillna(2)
matches["away_conf_strength"] = matches["away_confederation"].map(CONF_STRENGTH).fillna(2)
matches["conf_strength_gap"]  = matches["home_conf_strength"] - matches["away_conf_strength"]

# encode confederation for XGboost model 
from sklearn.preprocessing import LabelEncoder
le_conf = LabelEncoder()
all_confs = pd.concat([matches["home_confederation"], matches["away_confederation"]]).fillna("Unknown")
le_conf.fit(all_confs)
matches["home_confederation_enc"] = le_conf.transform(matches["home_confederation"].fillna("Unknown"))
matches["away_confederation_enc"] = le_conf.transform(matches["away_confederation"].fillna("Unknown"))

# convert bool to int
matches["is_competitive"] = matches["is_competitive"].astype(int)
matches["is_neutral"] = matches["neutral"].astype(int)

# Drop rows with no Elo or FIFA data at all (cannot build a meaningful feature row)
matches = matches.dropna(subset=["home_elo_rank", "away_elo_rank",
                                  "home_fifa_rank", "away_fifa_rank"])

# Calculate the difference between stats for home vs away
matches["elo_gap"] = matches["home_elo_rating"] - matches["away_elo_rating"]
matches["fifa_points_gap"] = matches["home_fifa_points"] - matches["away_fifa_points"]
matches["fifa_rank_gap"] = matches["home_fifa_rank"] - matches["away_fifa_rank"]
matches["same_confederation"] = (matches["home_confederation"] == matches["away_confederation"]).astype(int)

**Rolling form** — mean of the previous 5 matches, shifted so the current match is never included.
Home-side features use a team's previous home-side matches and away-side features its previous away-side matches.

In [13]:
# Rolling window features — the team's previous 5 matches on the same side (home or away)
WINDOW = 5

def add_rolling(df, stat_col, team_col, out_col, window=WINDOW):
    result = pd.Series(np.nan, index=df.index, dtype=float)
    for team, idx in df.groupby(team_col).groups.items():
        vals   = df.loc[idx, stat_col].astype(float)
        rolled = vals.shift(1).rolling(window, min_periods=2).mean()
        result.loc[idx] = rolled.values
    df[out_col] = result
    return df

# Cap individual match goals at 3 before rolling so outlier blowouts
# (e.g. 6-3, 5-0) don't inflate a team's expected-goals prior
matches["home_goals_clipped"] = matches["home_goals"].clip(upper=3)
matches["away_goals_clipped"] = matches["away_goals"].clip(upper=3)

# Goal-scoring & conceding form  ← use clipped columns here
matches = add_rolling(matches, "home_goals_clipped", "home_team", "home_scored_roll")
matches = add_rolling(matches, "away_goals_clipped", "home_team", "home_conceded_roll")
matches = add_rolling(matches, "away_goals_clipped", "away_team", "away_scored_roll")
matches = add_rolling(matches, "home_goals_clipped", "away_team", "away_conceded_roll")

# Defensive form (clean sheet rate)
matches = add_rolling(matches, "home_clean_sheet", "home_team", "home_cs_rate_roll")
matches = add_rolling(matches, "away_clean_sheet", "away_team", "away_cs_rate_roll")

# Cards
matches = add_rolling(matches, "home_yellows", "home_team", "home_yellows_roll")
matches = add_rolling(matches, "away_yellows", "away_team", "away_yellows_roll")
matches = add_rolling(matches, "home_reds",    "home_team", "home_reds_roll")
matches = add_rolling(matches, "away_reds",    "away_team", "away_reds_roll")

# Corners (separate helper to handle min_periods=1 for sparse early data)
def rolling_team_stat(df, team_col, stat_col, window=5):
    df  = df.sort_values("date").copy()
    out = {}
    for team, grp in df.groupby(team_col):
        rolled = grp[stat_col].shift(1).rolling(window, min_periods=1).mean()
        out.update(rolled.to_dict())
    return pd.Series(out)

GLOBAL_MEDIAN_CORNERS = matches["corners"].median()
matches["home_corners_roll"] = rolling_team_stat(matches, "home_team", "home_corners").fillna(GLOBAL_MEDIAN_CORNERS)
matches["away_corners_roll"] = rolling_team_stat(matches, "away_team", "away_corners").fillna(GLOBAL_MEDIAN_CORNERS)

# Red-card flag (binary) — reds are almost always 0 so a flag is cleaner
matches["home_red_likely"] = (matches["home_reds_roll"] > 0.1).astype(int)
matches["away_red_likely"] = (matches["away_reds_roll"] > 0.1).astype(int)

# Drop the intermediate binary columns used to compute rolling rates
matches.drop(columns=["home_win", "away_win", "home_clean_sheet", "away_clean_sheet"],
             inplace=True, errors="ignore")

matches.head(3)

,date,tournament,home_team,away_team,is_competitive,city,country,neutral,home_goals,away_goals,...,home_cs_rate_roll,away_cs_rate_roll,home_yellows_roll,away_yellows_roll,home_reds_roll,away_reds_roll,home_corners_roll,away_corners_roll,home_red_likely,away_red_likely
0,2024-01-01,Friendly,Japan,Thailand,0,Tokyo,Japan,False,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,0,0
1,2024-01-01,Friendly,China,Hong Kong,0,Abu Dhabi,United Arab Emirates,True,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,0,0
2,2024-01-02,Friendly,Indonesia,Libya,0,Antalya,Turkey,True,0,4,...,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,0,0


**Fill remaining gaps** — neutral priors for pairs without head-to-head history, and Elo-based priors for teams without enough rolling history.

In [14]:
# H2H: neutral priors when no history exists
H2H_NEUTRAL_FILLS = {
    "h2h_matches": 0,
    "h2h_win_rate": 1/3,
    "h2h_draw_rate": 1/3,
    "h2h_loss_rate": 1/3,
    "h2h_dominance": 0.3,
    "h2h_reliable": 0,
    "h2h_goals_f_per_match": 0,
    "h2h_goals_a_per_match": 0,
}
for col, fill_val in H2H_NEUTRAL_FILLS.items():
    if col in matches.columns:
        matches[col] = matches[col].fillna(fill_val)

# Rolling cold-start: early rows for a team have no rolling history.
# Use the team's Elo win-rate as a sensible prior instead of a global mean.
GLOBAL_MEDIAN_YELLOWS = matches["home_yellows"].median()

def cold_start_fill(df):
    df = df.copy()
    for side in ("home", "away"):
        wr = df[f"{side}_elo_win_rate"].fillna(0.4)
        df[f"{side}_scored_roll"]   = df[f"{side}_scored_roll"].fillna(wr * 2.5)
        df[f"{side}_conceded_roll"] = df[f"{side}_conceded_roll"].fillna((1 - wr) * 2.5)
        df[f"{side}_cs_rate_roll"]  = df[f"{side}_cs_rate_roll"].fillna(wr * 0.35)
        df[f"{side}_yellows_roll"]  = df[f"{side}_yellows_roll"].fillna(GLOBAL_MEDIAN_YELLOWS)
        df[f"{side}_reds_roll"]     = df[f"{side}_reds_roll"].fillna(0.0)
    return df

matches = cold_start_fill(matches)

## 6. Feature selection
1. **Collinearity** — highly correlated rating features add little new information.
2. **xG coverage** — check whether xG has enough real values to be useful.
3. **Permutation importance** — a proxy random forest is trained on the first 80% of matches; each feature is shuffled on the remaining 20% and the drop in accuracy is measured. Negative values mean shuffling the feature did not hurt accuracy.

In [15]:
# check for correlation between the ranks and ratings
rating_cols = [
    "home_elo_rank", "home_elo_rating", "home_fifa_rank", "home_fifa_points",
    "away_elo_rank", "away_elo_rating", "away_fifa_rank", "away_fifa_points",
]
corr = matches[rating_cols].corr().round(2)

print("Pairwise correlations among rating features (|r| > 0.85 → collinear):")
print("-" * 70)
for i, c1 in enumerate(rating_cols):
    for j, c2 in enumerate(rating_cols):
        if j <= i:
            continue
        r = corr.loc[c1, c2]
        if abs(r) > 0.85:
            flag = "⚠ COLLINEAR" if abs(r) > 0.94 else "  marginal"
            print(f"  {c1} x {c2}  r = {r:+.2f}  {flag}")

Pairwise correlations among rating features (|r| > 0.85 → collinear):
----------------------------------------------------------------------
  home_elo_rank x home_elo_rating  r = -0.99  ⚠ COLLINEAR
  home_elo_rank x home_fifa_rank  r = +0.98  ⚠ COLLINEAR
  home_elo_rank x home_fifa_points  r = -0.96  ⚠ COLLINEAR
  home_elo_rating x home_fifa_rank  r = -0.96  ⚠ COLLINEAR
  home_elo_rating x home_fifa_points  r = +0.96  ⚠ COLLINEAR
  home_fifa_rank x home_fifa_points  r = -0.99  ⚠ COLLINEAR
  away_elo_rank x away_elo_rating  r = -0.99  ⚠ COLLINEAR
  away_elo_rank x away_fifa_rank  r = +0.98  ⚠ COLLINEAR
  away_elo_rank x away_fifa_points  r = -0.96  ⚠ COLLINEAR
  away_elo_rating x away_fifa_rank  r = -0.96  ⚠ COLLINEAR
  away_elo_rating x away_fifa_points  r = +0.96  ⚠ COLLINEAR
  away_fifa_rank x away_fifa_points  r = -0.99  ⚠ COLLINEAR


In [16]:
xg_pct = raw_coverage["home_xG"]
print(f"xG observed in {xg_pct:.1f}% of matches before imputation "
      f"({100 - xg_pct:.1f}% filled with team/global medians).")

xG observed in 16.6% of matches before imputation (83.4% filled with team/global medians).


In [17]:
# Permutation importance on the validation set is a more honest signal:
# shuffle one feature at a time and measure how much accuracy drops.
# Negative = shuffling that feature IMPROVED accuracy (it was adding noise).

CANDIDATE_FEATURES = [
    # rating gaps 
    "elo_gap",
    "home_elo_win_rate", "away_elo_win_rate",
    # FIFA
    "fifa_rank_gap", "home_fifa_points", "away_fifa_points",
    # confederation
    "conf_strength_gap",
    "home_confederation_enc", "away_confederation_enc",
    # rolling form
    "home_scored_roll", "home_conceded_roll",
    "away_scored_roll", "away_conceded_roll",
    "home_cs_rate_roll", "away_cs_rate_roll",
    # h2h
    "h2h_matches", "h2h_win_rate", "h2h_draw_rate",
    "h2h_loss_rate", "h2h_dominance", "h2h_reliable",
    "h2h_goals_f_per_match", "h2h_goals_a_per_match",
    # match context
    "is_neutral",

    "home_xG_roll", "away_xG_roll", "xg_roll_gap",
    "home_win_rate_roll", "away_win_rate_roll",
    "same_confederation",
]

# Build xG roll columns (needed for the candidate check, will be dropped afterward)
matches = add_rolling(matches, "home_xG", "home_team", "home_xG_roll")
matches = add_rolling(matches, "away_xG", "away_team", "away_xG_roll")
matches["xg_roll_gap"] = matches["home_xG_roll"] - matches["away_xG_roll"]
matches["home_xG_roll"] = matches["home_xG_roll"].fillna(1.1)
matches["away_xG_roll"] = matches["away_xG_roll"].fillna(1.1)
matches["xg_roll_gap"] = matches["xg_roll_gap"].fillna(0.0)

# Temp win-rate rolls for the importance check
matches = add_rolling(matches, "h2h_win_rate", "home_team", "home_win_rate_roll")
matches = add_rolling(matches, "h2h_win_rate", "away_team", "away_win_rate_roll")
for side in ("home", "away"):
    wr = matches[f"{side}_elo_win_rate"].fillna(0.4)
    matches[f"{side}_win_rate_roll"] = matches[f"{side}_win_rate_roll"].fillna(wr)

# Train / val split (temporal — no data leakage)
matches = matches.sort_values("date").reset_index(drop=True)
split_idx = int(len(matches) * 0.80)
_train = matches.iloc[:split_idx].copy()
_val   = matches.iloc[split_idx:].copy()

_feats = [f for f in CANDIDATE_FEATURES if f in matches.columns]
_train_r = _train.dropna(subset=["winning_team"] + _feats)
_val_r   = _val.dropna(subset=["winning_team"] + _feats)

_le = LabelEncoder(); _le.fit(["away", "draw", "home"])
_y_train = _le.transform(_train_r["winning_team"])
_y_val   = _le.transform(_val_r["winning_team"])

_rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
_rf.fit(_train_r[_feats], _y_train)
print(f"Proxy RF val accuracy: {accuracy_score(_y_val, _rf.predict(_val_r[_feats])):.3f}")

_perm = permutation_importance(_rf, _val_r[_feats], _y_val, n_repeats=20, random_state=42, n_jobs=-1)
_perm_df = (pd.DataFrame({"feature": _feats,
                           "importance": _perm.importances_mean,
                           "std": _perm.importances_std})
            .sort_values("importance", ascending=False)
            .reset_index(drop=True))

print()
print(f"{'Feature':<35}  {'Importance':>10}  {'±Std':>8}  Decision")
print("-" * 72)
for _, row in _perm_df.iterrows():
    decision = (
        "KEEP   ✓" if row["importance"] > 0.001 else
        "TRIM   ~" if row["importance"] > 0 else
        "DROP   ✗"
    )
    print(f"  {row['feature']:<33}  {row['importance']:>+10.4f}  "
          f"{row['std']:>8.4f}  {decision}")

Proxy RF val accuracy: 0.715



Feature                              Importance      ±Std  Decision
------------------------------------------------------------------------
  elo_gap                               +0.0726    0.0116  KEEP   ✓
  h2h_dominance                         +0.0099    0.0048  KEEP   ✓
  h2h_win_rate                          +0.0098    0.0043  KEEP   ✓
  fifa_rank_gap                         +0.0090    0.0084  KEEP   ✓
  h2h_loss_rate                         +0.0084    0.0043  KEEP   ✓
  h2h_draw_rate                         +0.0078    0.0046  KEEP   ✓
  h2h_goals_f_per_match                 +0.0027    0.0029  KEEP   ✓
  h2h_matches                           +0.0015    0.0010  KEEP   ✓
  home_scored_roll                      +0.0013    0.0025  KEEP   ✓
  h2h_goals_a_per_match                 +0.0002    0.0013  TRIM   ~
  h2h_reliable                          +0.0000    0.0000  DROP   ✗
  conf_strength_gap                     -0.0007    0.0021  DROP   ✗
  away_cs_rate_roll                     -0

In [18]:
# Clean up the temporary columns used only for the analysis above.
for _col in ["home_xG_roll", "away_xG_roll", "xg_roll_gap",
             "home_win_rate_roll", "away_win_rate_roll"]:
    if _col in matches.columns:
        matches.drop(columns=[_col], inplace=True)
del _train, _val, _train_r, _val_r, _rf, _perm, _perm_df

### Chosen feature set
The feature set was chosen by hand, informed by the checks above:
- **Ratings:** the raw Elo and FIFA rank/rating columns are near-duplicates of each other (|r| ≥ 0.96), so the raw Elo columns are replaced by `elo_gap`, and FIFA is represented by `fifa_rank_gap` plus each team's FIFA points.
- **xG:** not used, because more than 80% of values are imputed.
- **Head-to-head:** win/draw/loss rates, the dominance score and the ≥4-matches reliability flag.
- **Form:** rolling goals scored for each side.
- **Context:** confederation-strength gap and neutral venue.

Apart from `elo_gap`, all permutation importances are below 0.01, and the final set does not follow them exactly:
`home_fifa_points`, `away_fifa_points`, `conf_strength_gap`, `h2h_reliable`, `away_scored_roll` and `is_neutral` are kept even though their importance was zero or negative.
The corners and cards models add their own rolling statistics on top of this base set.

In [19]:
# Shared base for result, score, and secondary models
BASE_FEATURES = [
    # Elo
    "elo_gap",

    # fifa
    "fifa_rank_gap", "home_fifa_points", "away_fifa_points",

    # confederation
    "conf_strength_gap",

    # head-to-head history
    "h2h_win_rate", "h2h_draw_rate",
    "h2h_loss_rate", "h2h_dominance", "h2h_reliable",

    # recent form
    "home_scored_roll",
    "away_scored_roll",

    # match context
    "is_neutral",
]

# Result and score models share the same features
RESULT_FEATURES = BASE_FEATURES[:]
SCORE_FEATURES = BASE_FEATURES[:]
RESULT_TARGET = "winning_team"
HOME_GOALS_TARGET = "home_goals"
AWAY_GOALS_TARGET = "away_goals"

# Corners adds its own rolling stats
CORNERS_FEATURES = BASE_FEATURES + ["home_corners_roll", "away_corners_roll"]
CORNERS_TARGET = "corners"

# Cards adds its own rolling stats + the competitive-match flag
CARDS_FEATURES = BASE_FEATURES + [
    "home_yellows_roll", "away_yellows_roll",
    "home_reds_roll", "away_reds_roll",
    "home_red_likely", "away_red_likely",
    "is_competitive",
]
HOME_YELLOWS_TARGET = "home_yellows"
AWAY_YELLOWS_TARGET = "away_yellows"
HOME_REDS_TARGET = "home_reds"
AWAY_REDS_TARGET = "away_reds"

## 7. Train / validation split
Chronological 80/20 split (no shuffling), so validation matches always come after training matches.

In [20]:
matches = matches.sort_values("date").reset_index(drop=True)
split_idx = int(len(matches) * 0.80)

train = matches.iloc[:split_idx].copy()
val   = matches.iloc[split_idx:].copy()
val_competitive = val[val["is_competitive"] == 1].copy()

print(f"Train: {len(train):,} rows  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Val:   {len(val):,} rows  ({val['date'].min().date()} → {val['date'].max().date()})")
print(f"Val (competitive only): {len(val_competitive):,} rows")

Train: 1,894 rows  (2024-01-01 → 2025-10-14)
Val:   474 rows  (2025-10-14 → 2026-06-04)
Val (competitive only): 110 rows


## 8. Hyperparameter tuning
Grid search with 5-fold `TimeSeriesSplit` cross-validation on the training set only.

In [21]:
# Tune result classifier 
train_result = train.dropna(subset=[RESULT_TARGET] + RESULT_FEATURES)
le_result    = LabelEncoder()
le_result.fit(["away", "draw", "home"])
y_result     = le_result.transform(train_result[RESULT_TARGET])

param_grid_result = {
    "max_depth": [3, 4, 5],
    "min_child_weight": [3, 5, 10],
    "learning_rate": [0.03, 0.05],
}

tscv = TimeSeriesSplit(n_splits=5)

gs_result = GridSearchCV(
    XGBClassifier(
        n_estimators=400, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0,
        eval_metric="mlogloss", num_class=3,
    ),
    param_grid_result,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
gs_result.fit(train_result[RESULT_FEATURES], y_result)
print(f"\nResult model — best params : {gs_result.best_params_}")
print(f"Result model — best CV acc : {gs_result.best_score_:.3f}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits



Result model — best params : {'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 10}
Result model — best CV acc : 0.636


In [22]:
# Tune home-goals regressor
train_score = train.dropna(subset=[HOME_GOALS_TARGET] + SCORE_FEATURES)

param_grid_reg = {
    "max_depth": [3, 4, 5],
    "min_child_weight": [3, 5, 10],
    "learning_rate": [0.03, 0.05],
}

gs_goals = GridSearchCV(
    XGBRegressor(
        n_estimators=400, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0, eval_metric="mae",
    ),
    param_grid_reg,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1,
)
gs_goals.fit(train_score[SCORE_FEATURES], train_score[HOME_GOALS_TARGET])
print(f"\nGoals model — best params : {gs_goals.best_params_}")
print(f"Goals model — best CV MAE : {-gs_goals.best_score_:.3f}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits



Goals model — best params : {'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 10}
Goals model — best CV MAE : 0.969


In [23]:
BEST_RESULT_PARAMS = gs_result.best_params_
BEST_GOALS_PARAMS  = gs_goals.best_params_

# Corners and cards models reuse the tuned result-model parameters (saves a separate search)
COMMON_PARAMS = dict(
    n_estimators=400, subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbosity=0,
    **BEST_RESULT_PARAMS,
)

## 9. Validation
One model per predicted quantity. `fit_models` is first trained on the **training split only** and scored on the
held-out validation split. (The original competition notebook scored models that had also been trained on the
validation rows, which inflated its accuracy.)

Baselines for the result model: *always predict a home win*, and *pick the team with the higher Elo rating*.
Corners and yellow-card errors are measured only on matches where those statistics were actually observed.

In [24]:
def fit_models(df):
    """Fit the result, goals, corners and cards models on df."""
    m = {}
    r = df.dropna(subset=[RESULT_TARGET] + RESULT_FEATURES)
    m["result"] = XGBClassifier(**{**COMMON_PARAMS, **BEST_RESULT_PARAMS}, eval_metric="mlogloss", num_class=3)
    m["result"].fit(r[RESULT_FEATURES], le_result.transform(r[RESULT_TARGET]))

    s = df.dropna(subset=[HOME_GOALS_TARGET, AWAY_GOALS_TARGET] + SCORE_FEATURES)
    for key, target in [("home_goals", HOME_GOALS_TARGET), ("away_goals", AWAY_GOALS_TARGET)]:
        m[key] = XGBRegressor(**{**COMMON_PARAMS, **BEST_GOALS_PARAMS}, eval_metric="mae")
        m[key].fit(s[SCORE_FEATURES], s[target])

    c = df.dropna(subset=[CORNERS_TARGET] + CORNERS_FEATURES)
    m["corners"] = XGBRegressor(**COMMON_PARAMS, eval_metric="mae")
    m["corners"].fit(c[CORNERS_FEATURES], c[CORNERS_TARGET])

    # Yellows as regressors; reds as binary classifiers (a red card is rare)
    k = df.dropna(subset=[HOME_YELLOWS_TARGET, AWAY_YELLOWS_TARGET] + CARDS_FEATURES)
    for key, target in [("home_yellows", HOME_YELLOWS_TARGET), ("away_yellows", AWAY_YELLOWS_TARGET)]:
        m[key] = XGBRegressor(**COMMON_PARAMS, eval_metric="mae")
        m[key].fit(k[CARDS_FEATURES], k[target])
    for key, target in [("home_reds", HOME_REDS_TARGET), ("away_reds", AWAY_REDS_TARGET)]:
        m[key] = XGBClassifier(**COMMON_PARAMS, eval_metric="logloss")
        m[key].fit(k[CARDS_FEATURES], (k[target] > 0).astype(int))
    return m


def evaluate(models, val_df, label):
    print(f"\n── {label} ({len(val_df)} rows) ──")
    vr = val_df.dropna(subset=[RESULT_TARGET] + RESULT_FEATURES)
    y_true = le_result.transform(vr[RESULT_TARGET])
    y_pred = models["result"].predict(vr[RESULT_FEATURES])
    print(f"  Result accuracy          : {accuracy_score(y_true, y_pred):.3f}")
    print(f"  Baseline — always home   : {(vr[RESULT_TARGET] == 'home').mean():.3f}")
    elo_pick = np.where(vr["elo_gap"] > 0, "home", "away")
    print(f"  Baseline — higher Elo    : {(elo_pick == vr[RESULT_TARGET]).mean():.3f}")
    print(classification_report(y_true, y_pred, target_names=le_result.classes_, zero_division=0))

    vs = val_df.dropna(subset=[HOME_GOALS_TARGET, AWAY_GOALS_TARGET] + SCORE_FEATURES)
    print(f"  Home goals MAE           : {mean_absolute_error(vs[HOME_GOALS_TARGET], models['home_goals'].predict(vs[SCORE_FEATURES])):.3f}")
    print(f"  Away goals MAE           : {mean_absolute_error(vs[AWAY_GOALS_TARGET], models['away_goals'].predict(vs[SCORE_FEATURES])):.3f}")

    vc = val_df[val_df["corners_observed"]]
    print(f"  Corners MAE (observed)   : {mean_absolute_error(vc[CORNERS_TARGET], models['corners'].predict(vc[CORNERS_FEATURES])):.3f}"
          f"  [n={len(vc)}]")
    vk = val_df[val_df["cards_observed"]]
    yel = models["home_yellows"].predict(vk[CARDS_FEATURES]) + models["away_yellows"].predict(vk[CARDS_FEATURES])
    print(f"  Yellow cards MAE (obs.)  : {mean_absolute_error(vk[HOME_YELLOWS_TARGET] + vk[AWAY_YELLOWS_TARGET], yel):.3f}"
          f"  [n={len(vk)}]")


val_models = fit_models(train)
evaluate(val_models, val, "Validation — all matches")
evaluate(val_models, val_competitive, "Validation — competitive matches only")


── Validation — all matches (474 rows) ──
  Result accuracy          : 0.719
  Baseline — always home   : 0.492
  Baseline — higher Elo    : 0.637
              precision    recall  f1-score   support

        away       0.73      0.74      0.73       129
        draw       0.56      0.39      0.46       112
        home       0.76      0.87      0.81       233

    accuracy                           0.72       474
   macro avg       0.68      0.67      0.67       474
weighted avg       0.71      0.72      0.71       474

  Home goals MAE           : 0.943
  Away goals MAE           : 0.845
  Corners MAE (observed)   : 2.785  [n=228]
  Yellow cards MAE (obs.)  : 1.739  [n=207]

── Validation — competitive matches only (110 rows) ──
  Result accuracy          : 0.700
  Baseline — always home   : 0.491
  Baseline — higher Elo    : 0.700
              precision    recall  f1-score   support

        away       0.70      0.78      0.74        36
        draw       0.33      0.10      0.15

## 10. Final models
After validation, the same models are refit on **all** matches for the tournament predictions.

In [25]:
models = fit_models(matches)

## 11. Team profiles and prediction function
Each team is represented by its feature values from the last match it appears in. `predict_match`:
- picks the winner from the result classifier (group stage: argmax over home/draw/away; knockout: higher of home/away after a FIFA-points adjustment);
- samples a scoreline from Poisson distributions whose means come from the goals regressors, resampling until it agrees with the chosen winner;
- flags penalties in knockout matches when the adjusted home/away probabilities are within 0.12;
- predicts corners, yellow cards and red cards from their own models.

In [26]:
# For each team, grab their feature values from the last match they appeared in
HOME_SIDE_COLS = {
    "home_confederation": "confederation",
    "home_conf_strength": "conf_strength",
    "home_scored_roll": "scored_roll",
    "home_yellows_roll": "yellows_roll",
    "home_reds_roll": "reds_roll",
    "home_corners_roll": "corners_roll",
    "home_red_likely": "red_likely",
    "home_elo_rating": "elo_rating",
    "home_fifa_rank": "fifa_rank",
    "home_fifa_points": "fifa_points"
}
AWAY_SIDE_COLS = {k.replace("home_", "away_"): v for k, v in HOME_SIDE_COLS.items()}

def build_team_profiles(df):
    profiles = {}
    for _, row in df.sort_values("date").iterrows():
        for side_map, team_col in [(HOME_SIDE_COLS, "home_team"),
                                    (AWAY_SIDE_COLS, "away_team")]:
            team = row.get(team_col)
            if pd.isna(team):
                continue
            profiles[team] = {clean: row.get(raw, np.nan)
                              for raw, clean in side_map.items()}
    return profiles

TEAM_PROFILES = build_team_profiles(matches)
print(f"Team profiles built for {len(TEAM_PROFILES)} teams")

Team profiles built for 210 teams


In [27]:
def get_profile(team):
    return TEAM_PROFILES.get(team)


def build_feature_row(home_team, away_team, is_neutral=True):
    hp = get_profile(home_team)
    ap = get_profile(away_team)

    h2h_defaults = {
        "h2h_win_rate":  1/3,
        "h2h_draw_rate": 1/3,
        "h2h_loss_rate": 1/3,
        "h2h_dominance": 0.3,
        "h2h_reliable":  0,
    }
    mask    = (h2h["team"] == home_team) & (h2h["opponent"] == away_team)
    h2h_row = h2h[mask]
    if len(h2h_row):
        r = h2h_row.iloc[0]
        h2h_defaults = {k: r.get(k, h2h_defaults[k]) for k in h2h_defaults}

    row = {
        "elo_gap":            hp["elo_rating"] - ap["elo_rating"],
        "home_fifa_points":     hp["fifa_points"],  
        "away_fifa_points":     ap["fifa_points"],
        "fifa_rank_gap":      hp["fifa_rank"]  - ap["fifa_rank"],
        "conf_strength_gap":  hp["conf_strength"] - ap["conf_strength"],
        "home_scored_roll":   hp["scored_roll"],
        "away_scored_roll":   ap["scored_roll"],
        "home_yellows_roll":  hp["yellows_roll"],
        "away_yellows_roll":  ap["yellows_roll"],
        "home_reds_roll":     hp["reds_roll"],
        "away_reds_roll":     ap["reds_roll"],
        "home_red_likely":    hp["red_likely"],
        "away_red_likely":    ap["red_likely"],
        "home_corners_roll":  hp["corners_roll"],
        "away_corners_roll":  ap["corners_roll"],
        "is_neutral":         int(is_neutral),
        "is_competitive":     1,
        **h2h_defaults,
    }
    return pd.DataFrame([row])


def predict_match(home_team, away_team, is_neutral=True, allow_draw=True, seed=None):
    """
    Predict a match result.

    The result classifier determines the winner via argmax of predicted probabilities
    (deterministic). Goals are then sampled from Poisson distributions parameterised
    by the regression model outputs, conditioned on that outcome — resampling until
    the scoreline agrees with the decided winner. This keeps the bracket stable while
    preserving natural score variety.

    Parameters
    ----------
    seed : int or None
        Optional per-call seed. The simulations below seed NumPy once
        (SEED) so the full set of predictions is reproducible.
    """
    if seed is not None:
        np.random.seed(seed)

    frow = build_feature_row(home_team, away_team, is_neutral)

    def reg(model, feats):
        cols = [c for c in feats if c in frow.columns]
        return float(model.predict(frow[cols])[0])

    def clf_proba(model, feats):
        cols = [c for c in feats if c in frow.columns]
        return model.predict_proba(frow[cols])[0]

    # ── Result probabilities ──────────────────────────────────────────────────
    probs     = clf_proba(models["result"], RESULT_FEATURES)
    classes   = list(le_result.classes_)
    away_prob = probs[classes.index("away")]
    draw_prob = probs[classes.index("draw")]
    home_prob = probs[classes.index("home")]

    # ── Sharpen probabilities using FIFA points gap ───────────────────────────
    # Used for the knockout winner and the penalty decision (the group-stage winner uses the
    # unadjusted argmax below). FIFA points range roughly 600–1900.
    # A gap of 200 points gives a nudge of ~0.08; 375+ points hits the 0.15 cap.
    fifa_points_gap = float(frow["home_fifa_points"].iloc[0]) - float(frow["away_fifa_points"].iloc[0])
    fifa_nudge      = np.clip(fifa_points_gap / 2500, -0.15, 0.15)
    home_prob       = np.clip(home_prob + fifa_nudge, 0.05, 0.90)
    away_prob       = np.clip(away_prob - fifa_nudge, 0.05, 0.90)
    total           = home_prob + draw_prob + away_prob
    home_prob      /= total
    draw_prob      /= total
    away_prob      /= total

    # ── Winner (deterministic — argmax of classifier probabilities) ───────────
    penalties = False
    if not allow_draw:
        # Knockout: ignore draw probability, pick the stronger side
        winner = "home" if home_prob >= away_prob else "away"
    else:
        winner = classes[int(np.argmax(probs))]

    # ── Scoreline (Poisson, conditioned on decided winner) ────────────────────
    lam_home = min(2.5, max(0.10, reg(models["home_goals"], SCORE_FEATURES)))
    lam_away = min(2.5, max(0.10, reg(models["away_goals"], SCORE_FEATURES)))

    MAX_TRIES = 50
    MAX_GOAL_DIFF = 3  # cap on the sampled winning margin
    for _ in range(MAX_TRIES):
        hg = int(np.random.poisson(lam_home))
        ag = int(np.random.poisson(lam_away))
        goal_result = "home" if hg > ag else ("away" if ag > hg else "draw")
        if goal_result == winner and abs(hg - ag) <= MAX_GOAL_DIFF:
            break
    else:
        if winner == "home":
            hg, ag = max(1, round(lam_home)), 0
        elif winner == "away":
            hg, ag = 0, max(1, round(lam_away))
        else:
            hg = ag = max(1, round((lam_home + lam_away) / 2))

    # ── Penalties (knockout only) ─────────────────────────────────────────────
    # Heuristic: if the adjusted home/away probabilities are within 0.12, predict a level
    # score after extra time and a penalty shootout.
    PENALTY_THRESHOLD = 0.12
    if not allow_draw and abs(home_prob - away_prob) < PENALTY_THRESHOLD:
        drawn_goals = min(2, int(np.random.poisson(min(lam_home, lam_away))))  # cap at 2 for realism
        hg, ag = drawn_goals, drawn_goals
        penalties = True

    # ── Secondary stats (rounding is fine here — no tie-breaking dependency) ─
    corners      = max(0, round(reg(models["corners"],      CORNERS_FEATURES)))
    home_yellows = max(0, round(reg(models["home_yellows"], CARDS_FEATURES)))
    away_yellows = max(0, round(reg(models["away_yellows"], CARDS_FEATURES)))
    home_reds    = int(clf_proba(models["home_reds"], CARDS_FEATURES)[1] > 0.5)
    away_reds    = int(clf_proba(models["away_reds"], CARDS_FEATURES)[1] > 0.5)

    return {
        "home_goals":    hg,
        "away_goals":    ag,
        "lam_home":      round(lam_home, 3),
        "lam_away":      round(lam_away, 3),
        "corners":       corners,
        "yellow_cards":  home_yellows + away_yellows,
        "red_cards":     home_reds + away_reds,
        "winning_team":  winner,
        "match_winner":  winner,
        "home_win_prob": home_prob,
        "draw_prob":     draw_prob,
        "away_win_prob": away_prob,
        "penalties":     penalties,
    }

## 12. Group stage simulation
Playoff placeholders in the DataCamp fixture file are replaced by the teams that qualified through the playoffs.
Groups are ranked by points, then goal difference, then goals scored.

In [28]:
group_fixtures  = pd.read_csv(DATA / "group_fixtures.csv")
knockout_slots  = pd.read_csv(DATA / "knockout_slots.csv")

MISSING_TEAMS = {
    "UEFA Playoff A": "Bosnia and Herzegovina",
    "UEFA Playoff B": "Sweden",
    "UEFA Playoff C": "Türkiye",
    "UEFA Playoff D": "Czechia",
    "FIFA Playoff 1": "DR Congo",
    "FIFA Playoff 2": "Iraq",
}

group_fixtures["home_team"] = group_fixtures["home_team"].replace(MISSING_TEAMS)
group_fixtures["away_team"] = group_fixtures["away_team"].replace(MISSING_TEAMS)
group_fixtures["home_team"] = group_fixtures["home_team"].apply(normalize_country_name)
group_fixtures["away_team"] = group_fixtures["away_team"].apply(normalize_country_name)

In [29]:
np.random.seed(SEED)  # makes the sampled scorelines reproducible

group_results = []
group_table   = {}

for _, fixture in group_fixtures.iterrows():
    home, away = fixture["home_team"], fixture["away_team"]
    match_id, group = int(fixture["match_id"]), fixture["group"]

    pred = predict_match(home, away, is_neutral=True, allow_draw=True)
    home_goals = pred["home_goals"]
    away_goals = pred["away_goals"]
    winner     = pred["winning_team"]

    home_pts = 3 if winner == "home" else (1 if winner == "draw" else 0)
    away_pts = 3 if winner == "away" else (1 if winner == "draw" else 0)

    for team, pts, gf, ga in [(home, home_pts, home_goals, away_goals),
                               (away, away_pts, away_goals, home_goals)]:
        if team not in group_table:
            group_table[team] = {"group": group, "points": 0,
                                  "goals_for": 0, "goals_against": 0, "goal_diff": 0}
        group_table[team]["points"] += pts
        group_table[team]["goals_for"] += gf
        group_table[team]["goals_against"] += ga
        group_table[team]["goal_diff"] += gf - ga

    group_results.append({
        "match_id": match_id,
        "group": group,
        "home_team": home,
        "away_team": away,
        "predicted_home_goals": home_goals,
        "predicted_away_goals": away_goals,
        "corners": pred["corners"],
        "yellow_cards": pred["yellow_cards"],
        "red_cards": pred["red_cards"],
        "winning_team": winner,
    })

In [30]:
standings_df = (
    pd.DataFrame.from_dict(group_table, orient="index")
    .reset_index().rename(columns={"index": "team"})
    .sort_values(["group", "points", "goal_diff", "goals_for"],
                 ascending=[True, False, False, False])
)
standings_df["pos"] = standings_df.groupby("group").cumcount() + 1

print("Group standings:")
for grp, sub in standings_df.groupby("group"):
    print(f"\n  Group {grp}")
    for _, row in sub.iterrows():
        marker = "✓" if row["pos"] <= 2 else " "
        print(f"    {marker} {int(row['pos'])}. {row['team']:30s}  "
              f"pts={int(row['points'])}  gd={int(row['goal_diff']):+d}  "
              f"gf={int(row['goals_for'])}")

Group standings:

  Group A
    ✓ 1. Czechia                         pts=6  gd=+4  gf=6
    ✓ 2. Mexico                          pts=6  gd=+1  gf=4
      3. South Korea                     pts=6  gd=+1  gf=4
      4. South Africa                    pts=0  gd=-6  gf=1

  Group B
    ✓ 1. Canada                          pts=7  gd=+4  gf=5
    ✓ 2. Switzerland                     pts=4  gd=-2  gf=2
      3. Qatar                           pts=3  gd=+1  gf=4
      4. Bosnia and Herzegovina          pts=3  gd=-3  gf=2

  Group C
    ✓ 1. Morocco                         pts=7  gd=+4  gf=5
    ✓ 2. Brazil                          pts=6  gd=+1  gf=4
      3. Scotland                        pts=4  gd=+0  gf=5
      4. Haiti                           pts=0  gd=-5  gf=2

  Group D
    ✓ 1. USA                             pts=6  gd=+1  gf=5
    ✓ 2. Paraguay                        pts=4  gd=+2  gf=5
      3. Türkiye                         pts=4  gd=-1  gf=3
      4. Australia                     

## 13. Knockout stage simulation
Group winners, runners-up and third-placed teams are mapped to the bracket slots. The 8 best third-placed teams are
assigned greedily (best-ranked eligible team first). This is a simplification of FIFA's official allocation table.

In [31]:
# Build slot_map: slot label → team name
slot_map = {}
for _, row in standings_df.iterrows():
    grp, pos, team = row["group"], int(row["pos"]), row["team"]
    if pos == 1:
        slot_map[f"Winner Group {grp}"]   = team
    elif pos == 2:
        slot_map[f"Runner-up Group {grp}"] = team
    elif pos == 3:
        slot_map[f"3rd Group {grp}"]       = team

# Assign Best 3rd slots — greedy, no repetition (simplified vs. FIFA's allocation table)
# Rank all 12 third-placers globally then consume one per slot
thirds_ranked = (
    standings_df[standings_df["pos"] == 3]
    .sort_values(["points", "goal_diff", "goals_for"], ascending=False)
    .reset_index(drop=True)
)

best3rd_slots = []
for col in ["slot_home", "slot_away"]:
    for label in knockout_slots[col].dropna():
        if str(label).startswith("Best 3rd") and label not in best3rd_slots:
            best3rd_slots.append(label)

used_teams = set()
for slot_label in best3rd_slots:
    m = re.search(r"\(Groups ([^)]+)\)", slot_label)
    if not m:
        slot_map[slot_label] = "Unknown"
        continue
    eligible   = [g.strip() for g in m.group(1).split("/")]
    candidates = thirds_ranked[
        thirds_ranked["group"].isin(eligible) & ~thirds_ranked["team"].isin(used_teams)
    ]
    if len(candidates):
        chosen = candidates.iloc[0]["team"]
        used_teams.add(chosen)
        slot_map[slot_label] = chosen
    else:
        slot_map[slot_label] = "Unknown"

print("Knockout slot → team:")
for k, v in sorted(slot_map.items()):
    print(f"  {k:50s} → {v}")

Knockout slot → team:
  3rd Group A                                        → South Korea
  3rd Group B                                        → Qatar
  3rd Group C                                        → Scotland
  3rd Group D                                        → Türkiye
  3rd Group E                                        → Côte d'Ivoire
  3rd Group F                                        → Sweden
  3rd Group G                                        → Egypt
  3rd Group H                                        → Cabo Verde
  3rd Group I                                        → Senegal
  3rd Group J                                        → Jordan
  3rd Group K                                        → Uzbekistan
  3rd Group L                                        → Panama
  Best 3rd (Groups A/B/C/D/F)                        → South Korea
  Best 3rd (Groups A/E/H/I/J)                        → Côte d'Ivoire
  Best 3rd (Groups B/E/F/I/J)                        → Qatar
  Best 3rd (Gro

In [32]:
knockout_results = []
match_winner_map = {}
match_loser_map  = {}

def resolve_slot(slot_label):
    if slot_label.startswith("Winner Match"):
        ref_id = int(slot_label.replace("Winner Match", "").strip())
        return match_winner_map.get(ref_id, "Unknown")
    if slot_label.startswith("Loser Match"):
        ref_id = int(slot_label.replace("Loser Match", "").strip())
        return match_loser_map.get(ref_id, "Unknown")
    return slot_map.get(slot_label, slot_label)

for _, slot in knockout_slots.iterrows():
    match_id   = int(slot["match_id"])
    round_name = slot["round"]
    home       = resolve_slot(slot["slot_home"])
    away       = resolve_slot(slot["slot_away"])

    pred   = predict_match(home, away, is_neutral=True, allow_draw=False)
    winner = home if pred["match_winner"] == "home" else away
    loser  = away if pred["match_winner"] == "home" else home

    match_winner_map[match_id] = winner
    match_loser_map[match_id]  = loser

    pens_str = ", pens" if pred["penalties"] else ""
    print(f"  {match_id:3d} ({round_name:22s}): "
          f"{home:25s} vs {away:25s} → {winner}  "
          f"({pred['home_goals']}-{pred['away_goals']}{pens_str})")

    knockout_results.append({
        "match_id":             match_id,
        "round":                round_name,
        "predicted_home_team":  home,
        "predicted_away_team":  away,
        "predicted_home_goals": pred["home_goals"],
        "predicted_away_goals": pred["away_goals"],
        "corners":              pred["corners"],
        "yellow_cards":         pred["yellow_cards"],
        "red_cards":            pred["red_cards"],
        "match_winner":         pred["match_winner"],
        "penalties":            pred["penalties"],
    })

   73 (Round of 32           ): Mexico                    vs Switzerland               → Switzerland  (0-3)
   74 (Round of 32           ): Morocco                   vs Netherlands               → Netherlands  (0-2)
   75 (Round of 32           ): Ecuador                   vs South Korea               → Ecuador  (1-0)
   76 (Round of 32           ): Japan                     vs Brazil                    → Brazil  (1-2)
   77 (Round of 32           ): Germany                   vs Norway                    → Germany  (2-0)
   78 (Round of 32           ): France                    vs Scotland                  → France  (3-1)
   79 (Round of 32           ): Czechia                   vs Cabo Verde                → Czechia  (2-0)


   80 (Round of 32           ): England                   vs Senegal                   → England  (2-2, pens)


   81 (Round of 32           ): Belgium                   vs Côte d'Ivoire             → Belgium  (0-0, pens)
   82 (Round of 32           ): USA                       vs Qatar                     → USA  (1-0)
   83 (Round of 32           ): Colombia                  vs Croatia                   → Croatia  (0-1)
   84 (Round of 32           ): Spain                     vs Austria                   → Spain  (1-0)


   85 (Round of 32           ): Canada                    vs Sweden                    → Canada  (3-1)
   86 (Round of 32           ): Paraguay                  vs Iran                      → Paraguay  (1-0)
   87 (Round of 32           ): Argentina                 vs Uruguay                   → Argentina  (2-1)
   88 (Round of 32           ): Portugal                  vs Panama                    → Portugal  (2-0)
   89 (Round of 16           ): Switzerland               vs Ecuador                   → Ecuador  (0-1)
   90 (Round of 16           ): Netherlands               vs Germany                   → Germany  (1-1, pens)


   91 (Round of 16           ): Brazil                    vs France                    → France  (0-1)


   92 (Round of 16           ): Czechia                   vs England                   → England  (0-3)
   93 (Round of 16           ): Croatia                   vs Spain                     → Spain  (0-3)
   94 (Round of 16           ): Belgium                   vs USA                       → Belgium  (2-1)
   95 (Round of 16           ): Paraguay                  vs Portugal                  → Portugal  (2-2, pens)


   96 (Round of 16           ): Canada                    vs Argentina                 → Argentina  (0-2)
   97 (Quarter-final         ): Ecuador                   vs Germany                   → Ecuador  (2-2, pens)
   98 (Quarter-final         ): Spain                     vs Belgium                   → Spain  (2-0)
   99 (Quarter-final         ): France                    vs England                   → France  (3-0)
  100 (Quarter-final         ): Portugal                  vs Argentina                 → Argentina  (2-3)
  101 (Semi-final            ): Ecuador                   vs Spain                     → Spain  (1-4)


  102 (Semi-final            ): France                    vs Argentina                 → Argentina  (0-2)


  103 (Third-place playoff   ): Ecuador                   vs France                    → France  (1-2)
  104 (Final                 ): Spain                     vs Argentina                 → Spain  (2-1)


## 14. Export predictions

In [33]:
results_lookup = {r["match_id"]: r for r in group_results}
group_predictions = group_fixtures.copy()
for col in ["predicted_home_goals", "predicted_away_goals",
            "corners", "yellow_cards", "red_cards", "winning_team"]:
    group_predictions[col] = group_predictions["match_id"].map(
        lambda mid, c=col: results_lookup[mid][c])

ko_lookup = {r["match_id"]: r for r in knockout_results}
knockout_predictions = knockout_slots.copy()
for col in ["predicted_home_team", "predicted_away_team",
            "predicted_home_goals", "predicted_away_goals",
            "corners", "yellow_cards", "red_cards",
            "match_winner", "penalties"]:
    knockout_predictions[col] = knockout_predictions["match_id"].map(
        lambda mid, c=col: ko_lookup[mid][c])

OUTPUTS.mkdir(exist_ok=True)
group_predictions.to_csv(OUTPUTS / "group_predictions.csv", index=False)
knockout_predictions.to_csv(OUTPUTS / "knockout_predictions.csv", index=False)
print(f"Group predictions:    {len(group_predictions)} rows → outputs/group_predictions.csv")
print(f"Knockout predictions: {len(knockout_predictions)} rows → outputs/knockout_predictions.csv")

Group predictions:    72 rows → outputs/group_predictions.csv
Knockout predictions: 32 rows → outputs/knockout_predictions.csv


In [34]:
print("Group predictions (first 10):")
display(group_predictions[[
    "match_id", "group", "home_team", "away_team",
    "predicted_home_goals", "predicted_away_goals",
    "corners", "yellow_cards", "red_cards", "winning_team",
]].head(10))

print("\nKnockout predictions:")
display(knockout_predictions[[
    "match_id", "round",
    "predicted_home_team", "predicted_away_team",
    "predicted_home_goals", "predicted_away_goals",
    "corners", "yellow_cards", "red_cards",
    "match_winner", "penalties",
]])

Group predictions (first 10):


,match_id,group,home_team,away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,1,0,8,3,0,home
1,2,A,South Korea,Czechia,1,0,10,4,0,home
2,3,B,Canada,Bosnia and Herzegovina,4,1,9,4,0,home
3,4,D,USA,Paraguay,2,1,10,4,0,home
4,5,D,Australia,Türkiye,2,0,7,4,0,home
5,6,B,Qatar,Switzerland,4,1,9,2,0,home
6,7,C,Brazil,Morocco,0,2,11,7,0,away
7,8,C,Haiti,Scotland,2,3,8,3,0,away
8,9,E,Germany,Curaçao,3,1,10,3,0,home
9,10,F,Netherlands,Japan,0,0,10,3,0,draw



Knockout predictions:


,match_id,round,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,Mexico,Switzerland,0,3,9,3,0,away,False
1,74,Round of 32,Morocco,Netherlands,0,2,9,5,0,away,False
2,75,Round of 32,Ecuador,South Korea,1,0,8,3,0,home,False
3,76,Round of 32,Japan,Brazil,1,2,10,3,0,away,False
4,77,Round of 32,Germany,Norway,2,0,10,3,0,home,False
5,78,Round of 32,France,Scotland,3,1,11,3,0,home,False
6,79,Round of 32,Czechia,Cabo Verde,2,0,9,2,0,home,False
7,80,Round of 32,England,Senegal,2,2,9,3,0,home,True
8,81,Round of 32,Belgium,Côte d'Ivoire,0,0,8,4,0,home,True
9,82,Round of 32,USA,Qatar,1,0,9,4,0,home,False
